In [1]:
from dotenv import load_dotenv
from langgraph.graph import StateGraph, START, END
from langchain_openrouter import ChatOpenRouter
from langchain_core.messages import HumanMessage
from typing import TypedDict

In [ ]:
load_dotenv()

In [ ]:
# Initialize the model
model = ChatOpenRouter(
    model="cohere/north-mini-code:free"
)

In [14]:
# state

class BlogState(TypedDict):

    title: str
    outline: str
    content: str
    evaluate: int

In [7]:
def create_outline(state: BlogState) -> BlogState:

    # fetch title
    title = state['title']


    # call llm
    prompt = f'Generate a detailed for a blog on the topic - {title}'
    outline = model.invoke(prompt).content


    # update state
    state['outline'] = outline

    return state

In [8]:
def create_blog(state: BlogState) -> BlogState:

    title = state['title']
    outline = state['outline']

    prompt = f'Write a detailed blog on the title - {title} using the following outline \n {outline}'
    content = model.invoke(prompt).content

    state['content'] = content

    return state

In [16]:
def evaluate(state: BlogState) -> BlogState:

    title = state['title']
    outline = state['outline']
    content = state['content']

    prompt = f'The Blog with title - {title} and with the based on this outline \n {outline} \n evaluate the blog \n {content} \n and only generate a integer score between 1 to 15'
    evauluation = model.invoke(prompt).content

    state['evaluate'] = evauluation
    return state

In [17]:
graph = StateGraph(BlogState)

#nodes
graph.add_node('create_outline', create_outline)
graph.add_node('create_blog', create_blog)
graph.add_node('evaluate', evaluate)


# edges
graph.add_edge(START, 'create_outline')
graph.add_edge('create_outline', 'create_blog')
graph.add_edge('create_blog', 'evaluate')
graph.add_edge('evaluate',END)

# compile
workflow = graph.compile()

In [ ]:
initial_state = {'title':'quaid azam'}

final_state = workflow.invoke(initial_state)